# ДЗ.3
Путь фотон испущен из источника с координатами (x,y,z) в направлении (nx,ny,nz). Рзаработать алгоритм, чтобы определить, попадет ли фотон в какой-либо фотоумножитель. Рассмотреть упрощенный случай:
    - пренебречь рассеянием и поглощением фотонов в сцинтилляторе
    - считать, что фотоны полностью поглощаются, достигая границы детектора

► Пусть количество фотоэлектронов, производимое фотокатодом, подчиняется распределению Пуассона со средним значением a. Реализовать метод, возвращающий случайное число фотоэлектронов. Какому классу должен принадлежать этот метод? Обоснуйте ваш выбор.


► Запустить моделирование для одного события с энергией 1 МэВ в центре детектора. Рассчитать полный собранный заряд (сумму фотоэлектронов по всем фотоумножителям). Насколько это число согласуется с ожидаемым Q = a*Y*(N*r**2/R**2)(тут опечатка, должно быть так Q = a*Y*(N*r**2/4R**2)?

In [39]:
import numpy as np

class Detector:
    def __init__(self, R: float, N: int, r: float, a: float):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

    def dots_on_parallels(self):
 # Разместим 40 фотоумножителей на сфере по 5 параллелям по 8 точек (5*8=40) на каждой, выбрав одинаковый шаг по cos(theta), а угол phi делим на 8 равных частей
       
        n_theta = 5
        n_phi = 8

        cos_theta = 1 - 2*(np.arange(n_theta) + 0.5)/n_theta
        theta = np.arccos(cos_theta)
        phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)

        dots = []

        for i in range(n_theta):
            for j in range(n_phi):
                x = self.R * np.sin(theta[i]) * np.cos(phi[j])
                y = self.R * np.sin(theta[i]) * np.sin(phi[j])
                z = self.R * np.cos(theta[i])

                dots.append([x, y, z])

        self.photocathode_centers = np.array(dots)


    def is_on_photocathode(self, coordinats):
        # Найдем расстояния от заданной точки до центров всех фотокатодов. Если хотя бы одно расстояние меньше радиуса фотокатода, возвращаем True

        lengths = np.linalg.norm(self.photocathode_centers - coordinats, axis=1)

        return np.any(lengths <= self.r)

#НОВАЯ ЧАСТЬ ДЛЯ ЭТОГО ЗАДАНИЯ - ДОБАВИЛА  методы intersection (находим точки пересечения траектории фотона с поверхностью детектора) и photoelectrons (разыгрываем число фотоэлектронов по распределению Пуассона со средним значением a)
    
    def intersection(self, coordinats, direction):
        projection = np.dot(coordinats, direction) # Проекция радиус вектора начальной точки на направление движения фотона

        distance = -projection + np.sqrt(projection**2 + self.R**2 - np.dot(coordinats, coordinats)) 
        # Расстояние от начальной точки до поверхности детектора

        point = coordinats + distance * direction

        return self.is_on_photocathode(point)

    # метод принадлежит Detector, а не Source, тк вероястность выбивания фотоэлектрона "а" уже есть в классе Detector
    def photoelectrons(self):

        return np.random.poisson(self.a)

detector = Detector(R=2.0, N=40, r=0.1, a=0.5)
detector.dots_on_parallels()

In [ ]:
import numpy as np
#тут все как было во втором задании
class Source:
    def __init__(self, Y: int):
        self.Y = Y

    def photon_directions(self, E: float):
    
        N = int(self.Y * E) #после вычисления кол-ва фотонов делаем его целым

# phi от 0 до 2pi, cosTheta от -1 до 1
        phi = np.random.uniform(0, 2*np.pi, N)
        cosTheta = np.random.uniform(-1, 1, N)
        sinTheta = np.sqrt(1 - cosTheta**2)

        x = sinTheta * np.cos(phi)
        y = sinTheta * np.sin(phi)
        z = cosTheta

        return np.array([x, y, z]).T


source = Source(Y=10000)


# проверка попадания (направление на центр фотокатода и направление на северный полюс, где фотокатода нет)

In [41]:
coordinats = np.array([0, 0, 0])

direction = detector.photocathode_centers[0] / detector.R
print(detector.intersection(coordinats, direction))

direction = np.array([0, 0, 1])
print(detector.intersection(coordinats, direction))


True
False


# проверяем генерацию числа фотоэлектронов

In [42]:
for i in range(15):
    print(detector.photoelectrons())

0
0
0
0
0
0
1
0
1
0
1
2
1
0
0


In [43]:
#третья часть 
source = Source(Y=10000)

coordinats = np.array([0, 0, 0])
directions = source.photon_directions(E=1.0)

Q = 0
n = 0 #подсчитываем количество попавших фотонов

for direction in directions:
    if detector.intersection(coordinats, direction):
        n += 1
        Q += detector.photoelectrons()

print("Количество фотонов, попавших на фотокатоды:", n)
print("Полный собранный заряд:", Q)


Количество фотонов, попавших на фотокатоды: 228
Полный собранный заряд: 116



# сравним полученный заряд с теоретическим значением (Q = a*Y*(N*r**2/4R**2))
 

In [44]:
Q_expected = detector.a * source.Y * detector.N * detector.r**2 / (4 * detector.R**2)

print("Полученный заряд:", Q)
print("Ожидаемый заряд:", round(Q_expected, 2))


Полученный заряд: 116
Ожидаемый заряд: 125.0


то есть при моделировании события с энергией 1 МэВ в центре детектора получили суммарный заряд 116 фотоэлектронов, а теоретическое значение 125 фотоэлектронов. Отличие в 9 фотоэлектронов невелико и связано со случайным характером генерации направлений фотонов и числа фотоэлектронов.